# 📝 Text Mining com Python I

Um projeto de **Text Mining**:

`Texto bruto → Limpeza (cleaning) → Representação numérica → Análise/Modelo`

**Como usar este notebook:** rode as células **em ordem, de cima para baixo**, observando a saída de cada uma antes de avançar. Comece pela instalação abaixo.

In [ ]:
# Instala as bibliotecas (execute uma vez; pode levar ~2 min no Colab)
!pip install -q spacy nltk scikit-learn plotly wordcloud
!python -m spacy download pt_core_news_sm -q
print("✅ Dependências instaladas.")

✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Dependências instaladas.


In [ ]:
import pandas as pd
import nltk
import numpy as np
import random
import re
import unicodedata

nltk.download("stopwords")
nltk.download("rslp")       # stemmer para português (RSLP)
nltk.download("punkt")
nltk.download("punkt_tab")  # necessário nas versões mais novas do NLTK

import plotly.express as px

random.seed(42)
np.random.seed(42)
print("✅ Ambiente pronto.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data] Error downloading 'rslp' from
[nltk_data]     <https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/packages/stemmers/rslp.zip>:   HTTP Error 429:
[nltk_data]     Too Many Requests
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Ambiente pronto.


## 1. O que é Text Mining?

**Text Mining** (mineração de texto) é o processo de extrair **informação útil**
de dados em **texto livre** (*unstructured text*): avaliações, e-mails, tweets,
tickets de suporte, notícias.

O grande obstáculo: modelos matemáticos e algoritmos trabalham com **números**,
não com palavras. Por isso, quase todo projeto de texto segue este **pipeline**:

`Texto bruto → Limpeza (cleaning) → Representação numérica → Análise/Modelo`

### Vocabulário essencial (em inglês, como no mercado)
- **Corpus**: o conjunto completo de textos que vamos analisar.
- **Document**: cada unidade de texto do corpus (ex.: uma avaliação).
- **Token**: cada pedaço mínimo do texto — geralmente uma palavra.
- **Vocabulary**: o conjunto de todos os tokens distintos do corpus.
- **Feature**: cada coluna numérica que representa o texto (ex.: a contagem de uma palavra).

Ao longo da aula, vamos percorrer esse pipeline inteiro com dados reais.

## 2. Introdução: o pipeline em ação

Antes de destrinchar cada etapa, vamos ver o **pipeline inteiro funcionando
em poucas linhas** — uma primeira degustação. A pergunta mais elementar do
text mining é: *quais palavras aparecem mais neste texto?* Respondê-la já
exige, em miniatura, quase todo o fluxo do curso:

`contar → identificar ruído → tokenizar → limpar → recontar`

Não se preocupe em dominar cada detalhe agora — cada peça terá seu módulo
próprio. A ideia é **enxergar o todo antes das partes**.

In [ ]:
from collections import Counter

# Texto de exemplo
texto = """A mineração de texto é uma técnica de processamento de linguagem natural
que permite extrair informações valiosas a partir de grandes volumes de textos."""

# Deixa tudo minúsculo, separa por espaços (split) e mantém só palavras
# "alfanuméricas" — isalnum() descarta tokens de pontuação solta

palavras = []                              # começa com lista vazia
for palavra in texto.lower().split():      # percorre cada palavra
    if palavra.isalnum():                  # se for alfanumérica...
        palavras.append(palavra)           # ...adiciona na lista

# em uma linha
# palavras = [palavra for palavra in texto.lower().split() if palavra.isalnum()]

# Counter conta quantas vezes cada palavra aparece
frequencia = Counter(palavras)

# As 5 palavras mais comuns
print(frequencia.most_common(10))

[('de', 5), ('a', 2), ('mineração', 1), ('texto', 1), ('é', 1), ('uma', 1), ('técnica', 1), ('processamento', 1), ('linguagem', 1), ('natural', 1)]


Resultado: `[('de', 5), ('a', 2), ('mineração', 1), ('texto', 1), ('é', 1)]`.

As campeãs são **`de`** e **`a`** — palavras sem conteúdo temático, que só
fazem a "cola" gramatical da frase. Elas dominam **qualquer** texto e escondem
o que de fato importa.

Essas palavras muito frequentes e pouco informativas têm nome: **stopwords**.
O NLTK já traz uma lista pronta para o português.

In [ ]:
from nltk.corpus import stopwords

stopwords_portugues = stopwords.words("portuguese")

print(f"O português tem {len(stopwords_portugues)} stopwords.")
print("As primeiras:", stopwords_portugues[:15])

O português tem 207 stopwords.
As primeiras: ['a', 'à', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'às', 'até', 'com', 'como', 'da']


### 2.1 Tokenizar e remover as stopwords

O `split()` é grosseiro: quebra só nos espaços e deixa a pontuação grudada
nas palavras. Um **tokenizer** de verdade separa as unidades com mais cuidado.
Vamos usar o **ToktokTokenizer** do NLTK (rápido e multilíngue), depois
remover as stopwords e contar de novo — agora só o que descreve o texto sobra.

> Um token é a menor unidade de texto processada por um modelo de linguagem. Dependendo do conteúdo, um token pode corresponder a uma palavra inteira, parte de uma palavra, um número, um sinal de pontuação ou até mesmo um espaço. Os modelos de inteligência artificial analisam e geram texto em tokens, e não em palavras ou caracteres, razão pela qual limites de contexto e custos de uso costumam ser medidos pela quantidade de tokens processados. Em média, um token equivale a aproximadamente quatro caracteres em inglês, embora essa relação varie conforme o idioma e a estrutura do texto.


https://gptforwork.com/tools/tokenizer

https://tokencalc.dev/tools/token-counter/

In [ ]:
from nltk.tokenize.toktok import ToktokTokenizer

# Tokenização usando ToktokTokenizer
tokenizer = ToktokTokenizer()
tokens = tokenizer.tokenize(texto)
print("Tokens:", tokens)
# repare: a pontuação final "." virou um token separado

Tokens: ['A', 'mineração', 'de', 'texto', 'é', 'uma', 'técnica', 'de', 'processamento', 'de', 'linguagem', 'natural', 'que', 'permite', 'extrair', 'informações', 'valiosas', 'a', 'partir', 'de', 'grandes', 'volumes', 'de', 'textos', '.']


In [ ]:
# Conjunto de stopwords (set deixa a checagem "está na lista?" mais rápida)
stopwords_portugues = set(stopwords.words("portuguese"))

# Mantém só os tokens que NÃO são stopwords (comparando em minúsculas)
tokens_filtrados = [token for token in tokens
                    if token.lower() not in stopwords_portugues]
print("Tokens após remoção de stopwords:\n", tokens_filtrados)

Tokens após remoção de stopwords:
 ['mineração', 'texto', 'técnica', 'processamento', 'linguagem', 'natural', 'permite', 'extrair', 'informações', 'valiosas', 'partir', 'grandes', 'volumes', 'textos', '.']


In [ ]:
# A mesma contagem de antes, agora sem o ruído das stopwords
frequencia_filtrada = Counter(tokens_filtrados)
print("Palavras mais comuns (sem stopwords):", frequencia_filtrada.most_common(5))

Palavras mais comuns (sem stopwords): [('mineração', 1), ('texto', 1), ('técnica', 1), ('processamento', 1), ('linguagem', 1)]


Compare o **antes** e o **depois**:

- **Antes** (contagem crua): as campeãs eram `de` e `a` — pura cola gramatical.
- **Depois** (sem stopwords): sobram `mineração`, `texto`, `técnica`,
  `processamento`... — agora só **palavras de conteúdo**, que revelam o
  assunto do texto.

Repare que ainda restou um `.` na lista filtrada: remover stopwords não cuida
de pontuação. Cada resíduo desses é justamente o que os próximos módulos
resolvem com técnicas de mercado.

Em ~20 linhas você viu o **esqueleto de todo projeto de text mining**. A
partir de agora vamos abrir cada etapa — começando por explorar textos com o
NLTK — e no fim aplicar tudo a um corpus real de avaliações e tweets.

## 3. Primeiros passos com o NLTK

O **NLTK** (*Natural Language Toolkit*) é a biblioteca mais tradicional de
NLP (*Natural Language Processing*) em Python — criada em 2001 e usada no
mundo inteiro para ensino e pesquisa. Ela acompanha um livro-texto clássico
(o *NLTK Book*) e traz **textos prontos para explorar**.

Antes de trabalhar com os nossos dados em português, vamos usar esse
"laboratório" para ganhar intuição: explorar contextos, contar palavras e
criar nossas primeiras métricas. Os textos são em **inglês** (Moby Dick,
discursos presidenciais americanos...) porque são o material didático
clássico da área — mas tudo o que fizermos aqui vale para qualquer idioma.

In [ ]:
# Parágrafo de exemplo
texto = """
A inteligência artificial está presente em diversas áreas do conhecimento.
Ela auxilia na análise de dados, na automação de tarefas e na criação de sistemas inteligentes.
Cada vez mais empresas utilizam essa tecnologia para melhorar seus processos.
"""

A função `sent_tokenize()` é utilizada para dividir um texto em frases. Ela identifica automaticamente onde uma frase termina, considerando sinais de pontuação (como ., ! e ?) e regras linguísticas.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

frases = sent_tokenize(texto, language='portuguese')
print(frases)

['\nA inteligência artificial está presente em diversas áreas do conhecimento.', 'Ela auxilia na análise de dados, na automação de tarefas e na criação de sistemas inteligentes.', 'Cada vez mais empresas utilizam essa tecnologia para melhorar seus processos.']


In [ ]:
palavras = word_tokenize(texto, language='portuguese')
print("Quantidade de palavras:", len(palavras))

Quantidade de palavras: 41


### 3.2 Nossos próprios tokens — em português do Brasil 🇧🇷

O mac_morpho é um corpus da língua portuguesa disponível na biblioteca NLTK. Ele reúne milhares de textos extraídos nos quais cada palavra foi previamente classificada com sua respectiva classe gramatical (substantivo, verbo, adjetivo, pronome, entre outras).

Por esse motivo, é amplamente utilizado em tarefas de Processamento de Linguagem Natural (PLN), como tokenização, etiquetagem morfossintática (Part-of-Speech Tagging), análise linguística e treinamento de modelos de aprendizado de máquina. Por ser um corpus oficial e amplamente mantido pela comunidade do NLTK, o mac_morpho é uma das principais opções para estudos e experimentos com textos em português.

In [ ]:
from nltk.corpus import mac_morpho
nltk.download("mac_morpho")

[nltk_data] Downloading package mac_morpho to /root/nltk_data...
[nltk_data]   Package mac_morpho is already up-to-date!


True

In [ ]:
mac_morpho.words()

['Jersei', 'atinge', 'média', 'de', 'Cr$', '1,4', ...]

In [ ]:
tokens = mac_morpho.words()

### 3.3 Enxergando o texto

**`.dispersion_plot(palavras)`** desenha **onde** cada palavra ocorre ao longo
do texto (cada traço = uma ocorrência). Usando os nomes das personagens, o
gráfico revela a **estrutura narrativa** do romance: quando cada uma entra e
sai da história.

### 3.4 Contando e medindo (len, set, sorted, count)

Boa parte do text mining é **contar** — e o Python puro já resolve muito:

- **`len(texto)`** → quantos tokens o texto tem (palavras **e** pontuação);
- **`set(texto)`** → o conjunto de tokens **distintos** (o *vocabulary*):
  `set` descarta as repetições;
- **`sorted(conjunto)`** → ordena o resultado;
- **`texto.count(palavra)`** → quantas vezes uma palavra específica aparece.

Vamos guardar cada medida numa **variável com nome claro** — hábito que vale
para o resto do curso.

In [ ]:
total_tokens = len(tokens)      # todos os tokens do romance
vocabulario = set(tokens)       # só os distintos (sem repetição)
total_distintos = len(vocabulario)

print("Total de tokens:  ", total_tokens)
print("Tokens distintos: ", total_distintos)

Total de tokens:   1170095
Tokens distintos:  67696


In [ ]:
# sorted() ordena o vocabulário — pontuação e Maiúsculas vêm primeiro
vocab_ordenado = sorted(vocabulario)

print("Início da lista: ", vocab_ordenado[:15])
print("Meio da lista:   ", vocab_ordenado[5000:5010])

Início da lista:  ['', '!', '"', '"Buddy"', '"Caminantes"', '"Chuck"', '"Cântico', '"Olga"', '"Parque', '"Polygram"', '"The', '"para', '$', '$%', '$&']
Meio da lista:    ['Algodoeiras', 'Algum', 'Alguma', 'Algumas', 'Algunas', 'Alguns', 'Alguém', 'Alhandra', 'Alheia', 'Alhures']


## 4. Os dados

Vamos trabalhar com **duas fontes** (como num projeto real), que estão na
pasta `data/` do projeto:

| Arquivo | Conteúdo | Formato |
|---|---|---|
| `avaliacoes_ecommerce.xlsx` | Avaliações de produtos com nota (1 a 5) | Excel |
| `tweets_lojatop.csv` | Tweets que mencionam a loja | CSV |
| `avaliacoes_restaurante.csv` | Base do desafio final | CSV |

No mundo real é assim mesmo: cada sistema exporta num formato diferente — o
site da loja gera um **Excel**, a coleta de redes sociais gera um **CSV**.
O `pandas` lê os dois com uma linha cada: `pd.read_excel()` e `pd.read_csv()`.

> 📤 **No Google Colab:** envie os **3 arquivos** da pasta `data/` e, em seguida, rode a célula abaixo.

In [ ]:
# Fonte 1 — E-commerce: Excel exportado do sistema da loja
df_ecom = pd.read_excel("avaliacoes_ecommerce.xlsx")

print(f"{len(df_ecom)} avaliações carregadas")
df_ecom.head()

250 avaliações carregadas


,id_avaliacao,data_compra,produto,texto_avaliacao,nota
0,1,2026-03-03,smartphone,"Adorei o smartphone, melhor compra do ano. Val...",4
1,2,2026-04-23,cafeteira,"Muito ruim, o cafeteira não funciona como anun...",1
2,3,2026-05-14,fone de ouvido,"Muito bom, recomendo demais! Qualidade impecável.",4
3,4,2026-03-02,fone de ouvido,"Muito ruim, o fone de ouvido não funciona como...",2
4,5,2026-04-23,teclado mecânico,"Péssimo, o teclado mecânico veio com defeito e...",1


In [ ]:
# Fonte 2 — Twitter: CSV com posts coletados que mencionam a @LojaTop
df_tweets = pd.read_csv("tweets_lojatop.csv")

print(f"{len(df_tweets)} tweets carregados")
df_tweets.head()

150 tweets carregados


,id_tweet,data_postagem,usuario,texto
0,10001,2026-04-10,@ju.oliveira,"chegou meu liquidificador e tô apaixonado, ent..."
1,10002,2026-01-28,@fer_nanda.c,Amei o novo teclado mecânico da @LojaTop!! Mel...
2,10003,2026-02-07,@camila_mts,"nunca mais compro na @LojaTop, smartphone horr..."
3,10004,2026-02-05,@fer_nanda.c,"nunca mais compro na @LojaTop, fone de ouvido ..."
4,10005,2026-01-18,@mari_reviews,"recebi o smartwatch, ainda testando pra formar..."


Repare: cada fonte tem **colunas diferentes** (`texto_avaliacao` vs. `texto`,
tweets não têm `nota`, o Excel tem `produto`...). Antes de juntar tudo num
único **corpus**, padronizamos: mesmos nomes de coluna e uma coluna `fonte`
para sabermos de onde cada documento veio.

In [ ]:
# Padroniza as colunas de cada fonte
ecom = df_ecom.rename(columns={"id_avaliacao": "id", "texto_avaliacao": "texto"})
ecom["fonte"] = "ecommerce"

tw = df_tweets.rename(columns={"id_tweet": "id"})
tw["fonte"] = "twitter"
tw["nota"] = np.nan            # tweet não tem nota

# Junta as duas fontes num único corpus (só as colunas que interessam)
colunas = ["id", "fonte", "texto", "nota"]
df = pd.concat([ecom[colunas], tw[colunas]], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # embaralha

print(f"Total de documentos no corpus: {len(df)}")
df.head(10)

Total de documentos no corpus: 400


,id,fonte,texto,nota
0,210,ecommerce,"Chegou no prazo. O mochila é mediano, nem bom ...",3.0
1,10031,twitter,Amei o novo cafeteira da @LojaTop!! Melhor com...,NaN
2,34,ecommerce,"Produto ok pelo preço, o teclado mecânico tem ...",3.0
3,211,ecommerce,"Ótimo custo-benefício, o liquidificador atende...",4.0
4,94,ecommerce,"Muito bom, recomendo demais! Qualidade impecável.",5.0
5,85,ecommerce,"Ótimo custo-benefício, o mochila atendeu tudo ...",5.0
6,10080,twitter,"Comprei na @LojaTop e não me arrependo, produt...",NaN
7,95,ecommerce,"Simplesmente perfeito, entrega rápida e atendi...",5.0
8,10017,twitter,"decepcionado com o teclado mecânico, esperava ...",NaN
9,127,ecommerce,"Estou muito satisfeito com o tênis de corrida,...",5.0


## 5. Exploração inicial (Exploratory Data Analysis)

Antes de processar, **olhe os dados**. Quantos documentos há por fonte?
Como se distribuem as notas? Qual o tamanho típico de um texto?

In [ ]:
# Raio-X do DataFrame: colunas, tipos e valores ausentes
# Repare nos "non-null": a coluna nota só existe para o e-commerce
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      400 non-null    int64  
 1   fonte   400 non-null    object 
 2   texto   400 non-null    object 
 3   nota    250 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 12.6+ KB


In [ ]:
contagem = df["fonte"].value_counts().reset_index()
contagem.columns = ["fonte", "quantidade"]

fig = px.bar(contagem, x="fonte", y="quantidade", text="quantidade",
             title="Documentos por fonte", color="fonte")
fig.show()

In [ ]:
notas = df_ecom["nota"].value_counts().sort_index().reset_index()
notas.columns = ["nota", "quantidade"]

fig = px.bar(notas, x="nota", y="quantidade", text="quantidade",
             title="Distribuição das notas (e-commerce)")
fig.show()

In [ ]:
df["n_caracteres"] = df["texto"].str.len()
df["n_palavras"] = df["texto"].str.split().apply(len)

fig = px.histogram(df, x="n_palavras", color="fonte", barmode="overlay",
                   title="Distribuição do nº de palavras por documento")
fig.show()

In [ ]:
# Quais produtos mais aparecem nas avaliações?
top_produtos = df_ecom["produto"].value_counts().reset_index()
top_produtos.columns = ["produto", "avaliacoes"]

fig = px.bar(top_produtos, x="avaliacoes", y="produto", orientation="h",
             text="avaliacoes", title="Produtos mais avaliados")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
# E como cada produto está avaliado? (nota média)
nota_media = (df_ecom.groupby("produto")["nota"].mean()
              .round(2).reset_index().sort_values("nota"))

fig = px.bar(nota_media, x="nota", y="produto", orientation="h",
             text="nota", range_x=[0, 5], title="Nota média por produto")
fig.show()

## 6. Limpeza (Text Cleaning / Preprocessing)

Texto bruto tem muito "ruído": maiúsculas/minúsculas misturadas, pontuação,
emojis, links, menções (@usuario). Normalizar reduz o **vocabulário** e ajuda
o modelo a focar no que importa.

Etapas típicas de **preprocessing** — vamos aplicar todas, uma de cada vez:
1. **Lowercasing** — deixar tudo minúsculo.
2. **Remover URLs**.
3. **Remover menções e hashtags** (comum em redes sociais).
4. **Remover acentos** (*accent stripping*) — opcional; simplifica, mas apaga distinções do português.
5. **Manter só letras** — remove pontuação, números e emojis.
6. **Normalizar espaços**.
7. **Remover stopwords** — palavras muito frequentes e pouco informativas ("de", "a", "que", "e").

> ⚠️ Cada etapa é uma **decisão**, não uma regra fixa. Remover acentos ajuda a
> agrupar erros de digitação, mas pode juntar palavras diferentes. Nesta aula
> removemos para simplificar — em produção, teste as duas opções.

**Estratégia:** em vez de sair aplicando tudo no DataFrame, vamos trabalhar com
**uma única frase** e ver o efeito de **cada** transformação. Só no final
juntamos tudo numa função e aplicamos ao corpus inteiro.

In [ ]:
# Nossa "cobaia": uma frase suja, típica de rede social.
# Ela vai nos acompanhar pelos módulos 6 a 10.
frase = "ADOREI o fone!!! 😍 Paguei R$ 150 na @LojaTop e chegou rápido: http://loja.top #recomendo"
print(frase)

ADOREI o fone!!! 😍 Paguei R$ 150 na @LojaTop e chegou rápido: http://loja.top #recomendo


### 6.1 Regex essencial (Regular Expressions)

Para limpar texto precisamos **encontrar padrões**: "qualquer URL", "qualquer
menção @usuario", "qualquer coisa que não seja letra". A ferramenta para isso
é o **regex** (*regular expression*): uma mini-linguagem para descrever padrões
de texto. Em Python, ela vive no módulo `re`.

Duas funções resolvem quase tudo:
- `re.findall(padrao, texto)` → **encontra** todas as ocorrências do padrão.
- `re.sub(padrao, substituto, texto)` → **substitui** as ocorrências por outra coisa.

Vamos aprender regex em **5 passos**, sempre testando na nossa frase.

In [ ]:
import re

# Passo 1 — Busca LITERAL: o padrão mais simples é o próprio texto
print(re.findall(r"fone", frase))   # procura exatamente "fone"
print(re.findall(r"xyz", frase))    # não existe -> lista vazia

['fone']
[]


In [ ]:
# Passo 2 — CLASSES DE CARACTERES: símbolos que representam "tipos" de caractere
#   \d -> um dígito (0-9)
#   \w -> um caractere "de palavra" (letra, dígito ou _)
#   \s -> um espaço em branco (espaço, tab, quebra de linha)
#   \S -> o CONTRÁRIO de \s (qualquer coisa que NÃO é espaço)

print("dígitos:   ", re.findall(r"\d", frase))
print("não-espaço:", re.findall(r"\S", frase)[:10], "...")  # os 10 primeiros

dígitos:    ['1', '5', '0']
não-espaço: ['A', 'D', 'O', 'R', 'E', 'I', 'o', 'f', 'o', 'n'] ...


In [ ]:
# Passo 3 — QUANTIFICADORES: quantas vezes o padrão se repete
#   +  -> UMA ou mais vezes (o mais usado na limpeza)
#   *  -> ZERO ou mais vezes

print(re.findall(r"\d", frase))    # sem quantificador: um dígito por vez
print(re.findall(r"\d+", frase))   # com +: dígitos consecutivos viram UM resultado

# Combinando classe + quantificador, os padrões ficam poderosos:
print(re.findall(r"@\w+", frase))    # "@" seguido de letras/dígitos = uma menção!
print(re.findall(r"http\S+", frase)) # "http" seguido de não-espaços = uma URL!

['1', '5', '0']
['150']
['@LojaTop']
['http://loja.top']


In [ ]:
# Passo 4 — CONJUNTOS [ ] e NEGAÇÃO [^ ]
#   [a-z]  -> qualquer letra minúscula entre a e z
#   [^a-z] -> o ^ NEGA o conjunto: qualquer coisa que NÃO seja letra minúscula

print(re.findall(r"[^a-z\s]", frase))
# Repare: maiúsculas, pontuação, emoji, dígitos... tudo que não é letra
# minúscula nem espaço. Por isso aplicaremos o lower() ANTES deste filtro!

['A', 'D', 'O', 'R', 'E', 'I', '!', '!', '!', '😍', 'P', 'R', '$', '1', '5', '0', '@', 'L', 'T', 'á', ':', ':', '/', '/', '.', '#']


In [ ]:
# Passo 5 — re.sub(padrao, substituto, texto): SUBSTITUI em vez de encontrar
print(re.sub(r"@\w+", "[USUARIO]", frase))              # troca a menção por um marcador
print()
print(re.sub(r"\s+", " ", "muitos    espaços     aqui"))  # colapsa espaços repetidos

ADOREI o fone!!! 😍 Paguei R$ 150 na [USUARIO] e chegou rápido: http://loja.top #recomendo

muitos espaços aqui


#### Os padrões da nossa limpeza (resumo)

| Padrão | Lê-se | Remove o quê |
|---|---|---|
| `http\S+` e `www\.\S+` | "http (ou www.) seguido de tudo que não é espaço" | URLs |
| `@\w+` | "@ seguido de um ou mais caracteres de palavra" | menções |
| `[^a-z\s]` | "tudo que NÃO é letra minúscula nem espaço" | pontuação, números, emojis |
| `\s+` | "um ou mais espaços seguidos" | espaços duplicados |

> 💡 Dois detalhes: na função final, os padrões de URL serão combinados com `|`,
> que em regex significa **OU** — `http\S+|www\.\S+`. E o `\.` é um ponto
> **literal**: sem a barra invertida, `.` em regex significa "qualquer caractere".

Pronto — esse é todo o regex de que precisamos. Agora vamos limpar a frase,
**uma etapa por célula**.

### 6.2 Limpeza passo a passo

Cada célula abaixo aplica **uma** transformação e mostra como a frase ficou.
Observe o que muda (e o que sobra) a cada passo.

In [ ]:
# Etapa 1 — Lowercasing: tudo minúsculo
# "ADOREI", "Adorei" e "adorei" devem contar como a MESMA palavra
etapa1 = frase.lower()
print(etapa1)

adorei o fone!!! 😍 paguei r$ 150 na @lojatop e chegou rápido: http://loja.top #recomendo


In [ ]:
# Etapa 2 — Remover URLs
# http\S+ ou www\.\S+ -> a URL inteira vira um espaço
etapa2 = re.sub(r"http\S+|www\.\S+", " ", etapa1)
print(etapa2)

adorei o fone!!! 😍 paguei r$ 150 na @lojatop e chegou rápido:   #recomendo


In [ ]:
# Etapa 3 — Remover menções e o símbolo de hashtag
# @\w+ -> a menção inteira some
# "#"  -> só o símbolo sai; a palavra fica ("recomendo" é informação útil!)
etapa3 = re.sub(r"@\w+", " ", etapa2)
etapa3 = re.sub(r"#", " ", etapa3)
print(etapa3)

adorei o fone!!! 😍 paguei r$ 150 na   e chegou rápido:    recomendo


In [ ]:
# Etapa 4 — Remover acentos (accent stripping)
# unicodedata.normalize("NFKD", ...) separa a letra do acento;
# combining() identifica o acento, e nós o descartamos
def remover_acentos(texto):
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

etapa4 = remover_acentos(etapa3)
print(etapa4)   # repare: "rápido" virou "rapido"

adorei o fone!!! 😍 paguei r$ 150 na   e chegou rapido:    recomendo


In [ ]:
# Etapa 5 — Manter apenas letras e espaços
# [^a-z\s] -> tudo que NÃO é letra minúscula nem espaço vira espaço
# (é aqui que pontuação, números e emojis desaparecem)
etapa5 = re.sub(r"[^a-z\s]", " ", etapa4)
print(etapa5)

adorei o fone      paguei r      na   e chegou rapido     recomendo


In [ ]:
# Etapa 6 — Normalizar espaços
# As remoções deixaram "buracos"; \s+ junta tudo num espaço só
etapa6 = re.sub(r"\s+", " ", etapa5).strip()
print(etapa6)

adorei o fone paguei r na e chegou rapido recomendo


In [ ]:
# Etapa 7 (preparação) — Stopwords: palavras muito comuns e pouco informativas
from nltk.corpus import stopwords

stopwords_pt = set(stopwords.words("portuguese"))
print(f"Total de stopwords em português: {len(stopwords_pt)}")
print(sorted(stopwords_pt)[:15])

Total de stopwords em português: 207
['a', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'até', 'com', 'como', 'da', 'das', 'de']


In [ ]:
# Etapa 7 — Remover stopwords (e palavras muito curtas)
# Como já removemos acentos da frase, removemos também das stopwords —
# senão "não" (com acento) nunca casaria com "nao" (sem acento)
stops = {remover_acentos(w) for w in stopwords_pt}

palavras = [w for w in etapa6.split() if w not in stops and len(w) > 2]
etapa7 = " ".join(palavras)
print(etapa7)

adorei fone paguei chegou rapido recomendo


In [ ]:
# 🎬 A evolução completa da frase, etapa por etapa:
historico = [("original    ", frase),  ("1 lowercase ", etapa1),
             ("2 urls      ", etapa2), ("3 menções   ", etapa3),
             ("4 acentos   ", etapa4), ("5 só letras ", etapa5),
             ("6 espaços   ", etapa6), ("7 stopwords ", etapa7)]
for nome, versao in historico:
    print(f"{nome} | {versao}")

original     | ADOREI o fone!!! 😍 Paguei R$ 150 na @LojaTop e chegou rápido: http://loja.top #recomendo
1 lowercase  | adorei o fone!!! 😍 paguei r$ 150 na @lojatop e chegou rápido: http://loja.top #recomendo
2 urls       | adorei o fone!!! 😍 paguei r$ 150 na @lojatop e chegou rápido:   #recomendo
3 menções    | adorei o fone!!! 😍 paguei r$ 150 na   e chegou rápido:    recomendo
4 acentos    | adorei o fone!!! 😍 paguei r$ 150 na   e chegou rapido:    recomendo
5 só letras  | adorei o fone      paguei r      na   e chegou rapido     recomendo
6 espaços    | adorei o fone paguei r na e chegou rapido recomendo
7 stopwords  | adorei fone paguei chegou rapido recomendo


### 6.3 Juntando tudo numa função

Agora que cada etapa está clara, empacotamos a sequência numa única função —
é ela que aplicaremos aos ~400 documentos do corpus.

In [ ]:
def limpar_texto(texto):
    texto = texto.lower()                            # 1. lowercasing
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)  # 2. remove URLs
    texto = re.sub(r"@\w+", " ", texto)              # 3. remove menções
    texto = re.sub(r"#", " ", texto)                 #    e o símbolo de hashtag
    texto = remover_acentos(texto)                   # 4. remove acentos
    texto = re.sub(r"[^a-z\s]", " ", texto)          # 5. mantém só letras
    texto = re.sub(r"\s+", " ", texto).strip()       # 6. normaliza espaços
    stops = {remover_acentos(w) for w in stopwords_pt}
    palavras = [w for w in texto.split()             # 7. remove stopwords
                if w not in stops and len(w) > 2]
    return " ".join(palavras)

# A função reproduz exatamente o nosso passo a passo?
print(limpar_texto(frase))
print("Igual à etapa 7?", limpar_texto(frase) == etapa7)

adorei fone paguei chegou rapido recomendo
Igual à etapa 7? True


In [ ]:
# Agora sim: aplicamos ao corpus inteiro
df["texto_limpo"] = df["texto"].apply(limpar_texto)
df[["texto", "texto_limpo"]].head(8)

,texto,texto_limpo
0,"Chegou no prazo. O mochila é mediano, nem bom ...",chegou prazo mochila mediano bom ruim
1,Amei o novo cafeteira da @LojaTop!! Melhor com...,amei novo cafeteira melhor compra mes recomendo
2,"Produto ok pelo preço, o teclado mecânico tem ...",produto preco teclado mecanico pros contras
3,"Ótimo custo-benefício, o liquidificador atende...",otimo custo beneficio liquidificador atendeu t...
4,"Muito bom, recomendo demais! Qualidade impecável.",bom recomendo demais qualidade impecavel
5,"Ótimo custo-benefício, o mochila atendeu tudo ...",otimo custo beneficio mochila atendeu tudo pre...
6,"Comprei na @LojaTop e não me arrependo, produt...",comprei arrependo produto qualidade
7,"Simplesmente perfeito, entrega rápida e atendi...",simplesmente perfeito entrega rapida atendimen...


## 7. Tokenização (Tokenization)

**Tokenization** é quebrar o texto em unidades menores — os **tokens**.
O caso mais comum é dividir por palavras (*word tokenization*).

Parece trivial (`texto.split()`), mas há sutilezas: pontuação colada na
palavra, contrações, símbolos. Por isso usamos tokenizadores prontos, como o
do **NLTK**. Compare as duas abordagens na nossa frase **original** (suja):

In [ ]:
from nltk.tokenize import word_tokenize

print("split():      ", frase.split()[:8])
print("word_tokenize:", word_tokenize(frase, language="portuguese")[:8])
# repare: o split() deixa "fone!!!" grudado; o NLTK separa a pontuação

split():       ['ADOREI', 'o', 'fone!!!', '😍', 'Paguei', 'R$', '150', 'na']
word_tokenize: ['ADOREI', 'o', 'fone', '!', '!', '!', '😍', 'Paguei']


In [ ]:
# No texto já LIMPO, a tokenização fica direta:
tokens_frase = word_tokenize(etapa7, language="portuguese")
print("Texto limpo:", etapa7)
print("Tokens:     ", tokens_frase)

Texto limpo: adorei fone paguei chegou rapido recomendo
Tokens:      ['adorei', 'fone', 'paguei', 'chegou', 'rapido', 'recomendo']


In [ ]:
# Aplicando ao corpus: cada documento vira uma lista de tokens
df["tokens"] = df["texto_limpo"].apply(lambda t: word_tokenize(t, language="portuguese"))
df[["texto_limpo", "tokens"]].head()

,texto_limpo,tokens
0,chegou prazo mochila mediano bom ruim,"[chegou, prazo, mochila, mediano, bom, ruim]"
1,amei novo cafeteira melhor compra mes recomendo,"[amei, novo, cafeteira, melhor, compra, mes, r..."
2,produto preco teclado mecanico pros contras,"[produto, preco, teclado, mecanico, pros, cont..."
3,otimo custo beneficio liquidificador atendeu t...,"[otimo, custo, beneficio, liquidificador, aten..."
4,bom recomendo demais qualidade impecavel,"[bom, recomendo, demais, qualidade, impecavel]"


## 8. Stemming

Palavras diferentes muitas vezes carregam a **mesma ideia de raiz**:
*"comprei", "comprou", "compraria"* — tudo vem de **comprar**. Para o
computador, porém, cada forma é um token diferente: isso **infla o
vocabulário** e espalha a contagem de um mesmo conceito por várias formas.

**Stemming** é a solução mais simples: **cortar o final da palavra** por
regras, mantendo só o radical (o **stem**). É rápido e bruto — o resultado
pode nem ser uma palavra real.

Antes de chamar qualquer biblioteca, vamos construir a intuição com Python puro.

In [ ]:
# A ideia na forma mais crua: CORTAR sufixos conhecidos do fim da palavra
def stem_caseiro(palavra):
    for sufixo in ["aria", "ando", "ei", "ou", "ar", "a", "s"]:
        if palavra.endswith(sufixo):
            return palavra[: -len(sufixo)]   # corta o sufixo
    return palavra                           # nenhum sufixo -> devolve inteira

palavra1 = "comprei"
palavra2 = "comprou"
palavra3 = "compraria"

print(palavra1, "->", stem_caseiro(palavra1))
print(palavra2, "->", stem_caseiro(palavra2))
print(palavra3, "->", stem_caseiro(palavra3))
# as três formas caíram no MESMO radical: "compr"

comprei -> compr
comprou -> compr
compraria -> compr


In [ ]:
# Por que isso importa? Nesta frase, o conceito "comprar" aparece 3 vezes:
avaliacao = "comprei ontem e compraria de novo pois quem compra aqui volta"
tokens_avaliacao = avaliacao.split()
stems_avaliacao = [stem_caseiro(t) for t in tokens_avaliacao]

print("Sem stemming:", tokens_avaliacao)
print("Com stemming:", stems_avaliacao)
# "comprei", "compraria" e "compra" agora contam como UMA coisa só ("compr").
# Mas repare no estrago: "pois" virou "poi"... regras ingênuas machucam palavras!

Sem stemming: ['comprei', 'ontem', 'e', 'compraria', 'de', 'novo', 'pois', 'quem', 'compra', 'aqui', 'volta']
Com stemming: ['compr', 'ontem', 'e', 'compr', 'de', 'novo', 'poi', 'quem', 'compr', 'aqui', 'volt']


Nosso stemmer caseiro tem 7 regras e já quebrou em "pois". Um stemmer de
verdade precisa de **muitas regras cuidadosas**, com exceções e ordem certa
de aplicação.

Para o português, o NLTK traz o **RSLP** (*Removedor de Sufixos da Língua
Portuguesa*), desenvolvido academicamente para a nossa língua — é ele que
usamos na prática.

In [ ]:
from nltk.stem import RSLPStemmer
import nltk
nltk.download("rslp")


stemmer = RSLPStemmer()   # RSLP: Removedor de Sufixos da Língua Portuguesa
variacoes = ["comprei", "comprou", "compraria", "compras",
             "recomendo", "recomendaria", "recomendam"]
for p in variacoes:
    print(f"{p:14} -> {stemmer.stem(p)}")
# repare: nem o RSLP é perfeito — "recomendo" virou "recom", mas
# "recomendaria" virou "recomend". Regras nunca cobrem tudo.

comprei        -> compr
comprou        -> compr
compraria      -> compr
compras        -> compr
recomendo      -> recom
recomendaria   -> recomend
recomendam     -> recomend


[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


In [ ]:
# E na nossa frase-cobaia (tokens do módulo 7):
stems_frase = [stemmer.stem(t) for t in tokens_frase]

print("Tokens:", tokens_frase)
print("Stems: ", stems_frase)
# rápido... mas legível? Stems são para a MÁQUINA, não para humanos.

Tokens: ['adorei', 'fone', 'paguei', 'chegou', 'rapido', 'recomendo']
Stems:  ['ador', 'fon', 'pag', 'cheg', 'rap', 'recom']


## 9. Lemmatization

O stemming **corta**; a lemmatization **consulta**. Em vez de remover sufixos
às cegas, ela usa um **dicionário** e a **análise gramatical** da frase para
achar o **lemma** — a forma canônica da palavra, a que está no dicionário:

- *comprei, comprou, compraria* → **comprar** (verbo no infinitivo)
- *produtos* → **produto** (substantivo no singular)
- *excelentes* → **excelente** (adjetivo no singular)

De novo: vamos construir a intuição na mão antes de usar biblioteca.

In [ ]:
# A ideia na forma mais crua: um DICIONÁRIO que mapeia forma -> lemma
dicionario_lemmas = {
    "comprei": "comprar", "comprou": "comprar", "compraria": "comprar",
    "chegaram": "chegar", "produtos": "produto", "excelentes": "excelente",
}

def lemma_caseiro(palavra):
    # devolve o lemma se a palavra estiver no dicionário; senão, ela mesma
    return dicionario_lemmas.get(palavra, palavra)

frase_lemma = "os produtos chegaram e são excelentes"
tokens_lemma = frase_lemma.split()
lemmas_caseiros = [lemma_caseiro(t) for t in tokens_lemma]

print("Tokens:", tokens_lemma)
print("Lemmas:", lemmas_caseiros)
# funcionou! mas... quem vai escrever o dicionário do português INTEIRO?

Tokens: ['os', 'produtos', 'chegaram', 'e', 'são', 'excelentes']
Lemmas: ['os', 'produto', 'chegar', 'e', 'são', 'excelente']


Dois problemas matam o dicionário caseiro:

1. **Escala** — o português tem centenas de milhares de formas (um único
   verbo tem dezenas de conjugações). Impossível manter na mão.
2. **Ambiguidade** — a mesma palavra pode ter lemmas diferentes conforme o
   contexto: em *"eu **gosto** de música"*, gosto é o verbo **gostar**; em
   *"o **gosto** do café"*, é o substantivo **gosto**.

Por isso lemmatizers de verdade, como o **spaCy**, combinam dicionário com um
**modelo estatístico** que analisa a frase inteira e decide a classe
gramatical (*POS — part of speech*) de cada palavra antes de escolher o lemma.

In [ ]:
import spacy
nlp = spacy.load("pt_core_news_sm")

# A mesma frase do lemma caseiro — agora com o spaCy (sem dicionário na mão)
doc = nlp("os produtos chegaram correndo e são excelentes")
for token in doc:
    print(f"{token.text:12} -> lemma: {token.lemma_:12} | classe: {token.pos_}")

os           -> lemma: o            | classe: DET
produtos     -> lemma: produto      | classe: NOUN
chegaram     -> lemma: chegar       | classe: VERB
correndo     -> lemma: correr       | classe: VERB
e            -> lemma: e            | classe: CCONJ
são          -> lemma: ser          | classe: AUX
excelentes   -> lemma: excelente    | classe: ADJ


In [ ]:
# O spaCy usa o CONTEXTO: nesta frase, "gosto" poderia ser substantivo
# e "canto" poderia ser verbo ("eu canto"). Veja como ele decide:
doc = nlp("eu gosto do canto do pássaro")
for token in doc:
    print(f"{token.text:10} -> lemma: {token.lemma_:10} | classe: {token.pos_}")
# "gosto" -> gostar (VERB) e "canto" -> canto (NOUN): decisão pelo contexto!
# curiosidade: "do" vira "de o" — o spaCy desfaz a contração.
# (usamos o modelo "sm", o menor; os modelos md/lg erram menos)

eu         -> lemma: eu         | classe: PRON
gosto      -> lemma: gostar     | classe: VERB
do         -> lemma: de o       | classe: ADP
canto      -> lemma: canto      | classe: NOUN
do         -> lemma: de o       | classe: ADP
pássaro    -> lemma: pássaro    | classe: NOUN


## 10. Stemming vs. Lemmatization: qual usar?

| | **Stemming** | **Lemmatization** |
|---|---|---|
| Como funciona | corta sufixos por regras | dicionário + análise gramatical |
| Resultado | radical, pode não ser palavra real ("compr") | palavra real ("comprar") |
| Velocidade | ⚡ muito rápido | 🐢 mais lento (analisa a frase toda) |
| Precisão | menor — às vezes agressivo demais | maior — entende o contexto |
| Ferramenta | NLTK (`RSLPStemmer`) | spaCy (`token.lemma_`) |

**Quando usar cada um:**
- **Stemming** → bases enormes, protótipos rápidos, busca/indexação
  (motores de busca usam muito), quando ninguém vai ler o resultado.
- **Lemmatization** → análises que humanos vão ler (nuvens de palavras,
  top termos), quando a precisão importa e o corpus tem tamanho gerenciável.
- Na dúvida, **teste os dois** e compare no seu problema.

**Nesta aula** seguiremos com o texto **lematizado**: nosso corpus é pequeno
e queremos gráficos legíveis.

In [ ]:
# A mesma frase pelos dois caminhos:
frase_comparacao = "os produtos chegaram correndo e são excelentes"

caminho_stem = [stemmer.stem(t) for t in frase_comparacao.split()]
caminho_lemma = [t.lemma_ for t in nlp(frase_comparacao)]

print("Original:", frase_comparacao.split())
print("Stemming:", caminho_stem)
print("Lemmas:  ", caminho_lemma)
# stems: rápidos, mas ilegíveis ("produt", "excel")
# lemmas: palavras de verdade ("produto", "excelente")

Original: ['os', 'produtos', 'chegaram', 'correndo', 'e', 'são', 'excelentes']
Stemming: ['os', 'produt', 'cheg', 'corr', 'e', 'são', 'excel']
Lemmas:   ['o', 'produto', 'chegar', 'correr', 'e', 'ser', 'excelente']


In [ ]:
def lematizar(texto):
    doc = nlp(texto)
    return " ".join([t.lemma_ for t in doc if not t.is_space])

# Aplicando ao corpus inteiro (pode levar ~1 min).
# Obs.: nosso texto_limpo está SEM acentos, o que derruba um pouco a
# qualidade dos lemmas — troca aceitável aqui; em produção, lematize
# ANTES de remover os acentos.
df["texto_lema"] = df["texto_limpo"].apply(lematizar)
df[["texto_limpo", "texto_lema"]].head()

,texto_limpo,texto_lema
0,chegou prazo mochila mediano bom ruim,chegar prazo mochila mediano bom ruim
1,amei novo cafeteira melhor compra mes recomendo,amei novo cafeteira bom compra mes recomendo
2,produto preco teclado mecanico pros contras,produto preco teclado mecanico pro contra
3,otimo custo beneficio liquidificador atendeu t...,otimo custo beneficio liquidificador atender t...
4,bom recomendo demais qualidade impecavel,bom recomendo demais qualidade impecavel
